In [100]:
import numpy as np

In [101]:
from huggingface_hub import login
from huggingface_hub import whoami
# Use to login to huggingface
login()

# Check Login Info
user = whoami(token=...)
print(user)

{'type': 'user', 'id': '6ad91bff8b4f267e96f60c0c', 'name': 'einobe13', 'fullname': 'Elizabeth Thomas', 'canPay': False, 'billingMode': 'prepaid', 'periodEnd': 1793491200, 'isPro': False, 'avatarUrl': '/avatars/a6ce4e1bd5a4beb498cb5960aeb44f87.svg', 'orgs': [], 'auth': {'type': 'oauth', 'expiresAt': '2026-11-20T20:23:03.000Z'}}


In [5]:
import torch

print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

False


AssertionError: Torch not compiled with CUDA enabled

In [121]:
from datasets import load_dataset

df = load_dataset("SetFit/mrpc")

# from datasets import load_dataset

glue_df = load_dataset("nyu-mll/glue", "ax")

Repo card metadata block was not found. Setting CardData to empty.


In [123]:
print(f"These are the types of values in the glue dataset that I'M NOT USING:\n",glue_df._check_values_type,"\n")

These are the types of values in the glue dataset that I'M NOT USING:
 <bound method DatasetDict._check_values_type of DatasetDict({
    test: Dataset({
        features: ['premise', 'hypothesis', 'label', 'idx'],
        num_rows: 1104
    })
})> 



### There are 3,668 pairs of sentences in the training set, 408 in the validation set, and 1,725 in the test set

In [103]:
# remove 'label_text' from every split in-place
df = df.remove_columns('label_text')

print(f"These are the types of values in the dataset:\n",df._check_values_type,"\n")

These are the types of values in the dataset:
 <bound method DatasetDict._check_values_type of DatasetDict({
    train: Dataset({
        features: ['text1', 'text2', 'label', 'idx'],
        num_rows: 3668
    })
    validation: Dataset({
        features: ['text1', 'text2', 'label', 'idx'],
        num_rows: 408
    })
    test: Dataset({
        features: ['text1', 'text2', 'label', 'idx'],
        num_rows: 1725
    })
})> 



# Preprocessing

#### In the **Train dataset** The labels are already integers, so preprocessing is not necessary

#### For the **'label'** attribute the values are:
    - 0: not_equivalent
    - 1: equivalent

In [104]:
train_df = df["train"]
valid_df = df["validation"]
test_df = df["test"]

print(f"First row of the train dataset has an 'equivalent' label:")
print(train_df[0])
print(f"Value Types in the Train Dataset")
print(train_df.features)

print(f"the 87th row of the Validation dataset has an 'not_equivalent' label:")
print(valid_df[87])
print(f"Value Types in the Validation Dataset")
print(valid_df.features)

print(f"The 15th row of the Test dataset has an 'not_equivalent' label:")
print(test_df[15])
print(f"Value Types in the Test Dataset")
print(test_df.features)

First row of the train dataset has an 'equivalent' label:
{'text1': 'Amrozi accused his brother , whom he called " the witness " , of deliberately distorting his evidence .', 'text2': 'Referring to him as only " the witness " , Amrozi accused his brother of deliberately distorting his evidence .', 'label': 1, 'idx': 0}
Value Types in the Train Dataset
{'text1': Value('string'), 'text2': Value('string'), 'label': Value('int64'), 'idx': Value('int64')}
the 87th row of the Validation dataset has an 'not_equivalent' label:
{'text1': 'However , EPA officials would not confirm the 20 percent figure .', 'text2': 'Only in the past few weeks have officials settled on the 20 percent figure .', 'label': 0, 'idx': 812}
Value Types in the Validation Dataset
{'text1': Value('string'), 'text2': Value('string'), 'label': Value('int64'), 'idx': Value('int64')}
The 15th row of the Test dataset has an 'not_equivalent' label:
{'text1': 'Ballmer has been vocal in the past warning that Linux is a threat to 

# Tokenize the dataset text
#### We need to convert the text to numbers the model can make sense of. This is done with a tokenizer
> #### **WHY?**: Models expect numerical tensors, not raw text strings. Keeping text columns would cause errors.

- **'input_ids'**: contains two rows of integers (one for each sentence) that are the unique identifiers of the tokens in each sentence.
- **'token_type_ids'**: tells the model which part of the input is the the first sentence (0) and second sentence (1) in sentence **pair** tasks.
- **'attention_mask'**: tensors with the exact same shape as the input IDs tensor, filled with 0s and 1s: 1s indicate the corresponding tokens should be attended to, and 0s indicate the corresponding tokens should not be attended to (i.e., they should be ignored by the attention layers of the model).

In [105]:
# pip install transformers

When analyzing the 15th row in the 'train' dataset, tokenizing text1 and text2 separately, then tokenizing text1 & text2 together we can see that in the paired tokenization:

**The parts of the input corresponding to the 'text1' have token_type_id = 0, while 'text2' have token_type_id = 1**

Example:

Tokenized_pair input_ids: [101, 24049, 2001, 2087, 3728, 3026, 3580, 2343, 2005, 1996, 9722, 1004, 4132, 9340, 12439, 2964, 2449, 1012, 102, 3026, 3580, 2343, 4388, 24049, 1010, 3839, 2132, 1997, 1996, 9722, 1998, 4132, 9340, 12439, 2964, 3131, 1010, 2097, 2599, 1996, 2047, 9178, 1012, 102]

Tokenized_pair attention_mask: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]

tokenized_pair token_type_ids: [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]

Tokens (pair): ['[CLS]', 'rudder', 'was', 'most', 'recently', 'senior', 'vice', 'president', 'for', 'the', 'developer', '&', 'platform', 'evan', '##gel', '##ism', 'business', '.', '[SEP]', 'senior', 'vice', 'president', 'eric', 'rudder', ',', 'formerly', 'head', 'of', 'the', 'developer', 'and', 'platform', 'evan', '##gel', '##ism', 'unit', ',', 'will', 'lead', 'the', 'new', 'entity', '.', '[SEP]']

> In general, you don’t need to worry about whether or not there are token_type_ids in your tokenized inputs: as long as you use the same checkpoint for the tokenizer and the model, everything will be fine as the tokenizer knows what to provide to its mode


In [106]:
from transformers import AutoTokenizer

checkpoint = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

text1 = df["train"]["text1"]
text2 = df["train"]["text2"]

# Tokenize the train split texts (pass lists; enable truncation/padding as needed)
tokenized_sentences_1 = tokenizer(list(text1), padding=False, truncation=True)
tokenized_sentences_2 = tokenizer(list(text2), padding=False, truncation=True)

# tokenize as a pair (sentence-pair encoding)
tokenized_pair = tokenizer(list(text1), list(text2), padding=False, truncation=True)

# index to inspect
idx = 15

# raw texts
print("Text1:", text1[idx])
print("Text2:", text2[idx])

# tokenized dict keys
# print("\nTokenized_sentences_1 keys:\n", tokenized_sentences_1.keys())

# raw input ids and attention masks for the 15th example
print("\nTokenized_sentences_1 input_ids:", tokenized_sentences_1['input_ids'][idx])
print("Tokenized_sentences_1 attention_mask:", tokenized_sentences_1.get('attention_mask', None) and tokenized_sentences_1['attention_mask'][idx])
print("Tokenized_sentences_1 token_type_ids:", tokenized_sentences_1['token_type_ids'][idx])

print("\nTokenized_sentences_2 input_ids:", tokenized_sentences_2['input_ids'][idx])
print("Tokenized_sentences_2 attention_mask:", tokenized_sentences_2.get('attention_mask', None) and tokenized_sentences_2['attention_mask'][idx])
print("Tokenized_sentences_2 token_type_ids:", tokenized_sentences_2['token_type_ids'][idx])

print("\nTokenized_pair input_ids:", tokenized_pair['input_ids'][idx])
print("Tokenized_pair attention_mask:", tokenized_pair.get('attention_mask', None) and tokenized_pair['attention_mask'][idx])
print("tokenized_pair token_type_ids:", tokenized_pair['token_type_ids'][idx])

# readable token lists
print("\nTokens (text1):", tokenizer.convert_ids_to_tokens(tokenized_sentences_1['input_ids'][idx]))
print("Tokens (text2):", tokenizer.convert_ids_to_tokens(tokenized_sentences_2['input_ids'][idx]))
print("Tokens (pair):", tokenizer.convert_ids_to_tokens(tokenized_pair['input_ids'][idx]))

# decoded strings (skip special tokens)
print("\nDecoded text1:", tokenizer.decode(tokenized_sentences_1['input_ids'][idx], skip_special_tokens=True))
print("Decoded text2:", tokenizer.decode(tokenized_sentences_2['input_ids'][idx], skip_special_tokens=True))
print("Decoded pair (concatenated):", tokenizer.decode(tokenized_pair['input_ids'][idx], skip_special_tokens=True))


Text1: Rudder was most recently senior vice president for the Developer & Platform Evangelism Business .
Text2: Senior Vice President Eric Rudder , formerly head of the Developer and Platform Evangelism unit , will lead the new entity .

Tokenized_sentences_1 input_ids: [101, 24049, 2001, 2087, 3728, 3026, 3580, 2343, 2005, 1996, 9722, 1004, 4132, 9340, 12439, 2964, 2449, 1012, 102]
Tokenized_sentences_1 attention_mask: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
Tokenized_sentences_1 token_type_ids: [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]

Tokenized_sentences_2 input_ids: [101, 3026, 3580, 2343, 4388, 24049, 1010, 3839, 2132, 1997, 1996, 9722, 1998, 4132, 9340, 12439, 2964, 3131, 1010, 2097, 2599, 1996, 2047, 9178, 1012, 102]
Tokenized_sentences_2 attention_mask: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
Tokenized_sentences_2 token_type_ids: [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 

## What happens when padding is included or not?

In [ ]:
#This function takes a dictionary (like the items of our dataset) and returns a new dictionary with additional keys: 
    # input_ids, attention_mask, and token_type_ids

# NOTE: the padding argument is removed because:
    # padding all the samples to the maximum length is not efficient: 
    # it’s better to pad the samples when we’re building a batch

def tokenize_function(dataset):
    return tokenizer(dataset["text1"], dataset["text2"], truncation=True)

### The **Dataset.Map()** Function can be used on large datasets!
#### Processing in batches allows the fast tokenizer to work on multiple examples simultaneously, significantly improving speed

In [ ]:
# use the function to map the entire dataset.
# this adds new columns to the dataset
tokenized_datasets = df.map(tokenize_function, batched=True)
tokenized_datasets

Map: 100%|██████████| 1725/1725 [00:00<00:00, 7460.03 examples/s]


DatasetDict({
    train: Dataset({
        features: ['text1', 'text2', 'label', 'idx', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 3668
    })
    validation: Dataset({
        features: ['text1', 'text2', 'label', 'idx', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 408
    })
    test: Dataset({
        features: ['text1', 'text2', 'label', 'idx', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 1725
    })
})

## What about **Dynamic Padding**?
#### Dynamic padding means the samples in the batch (entries 0-8) should all be padded to a length of 67, the maximum length inside the batch. 
#### Without dynamic padding, all of the samples would have to be padded to the maximum length in the *whole dataset*, or the maximum length the model can accept.

In [116]:
from transformers import DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# analyze the lengths of each entry in the batch:
samples = tokenized_datasets["train"][0:8] # entries 0-8
samples = {k: v for k, v in samples.items() if k not in ["idx", "sentence1", "sentence2"]} # remove columns
[len(x) for x in samples["input_ids"]]

[50, 59, 47, 67, 59, 50, 62, 32]

# Fine-tuning the model